In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

def analyze_files(file_paths, 
                  x_col='Hemorrhage Rate %/Time', 
                  y_cols=None):
    """
    Analyze multiple Excel files with curve fits (linear, quadratic, exponential).

    Parameters
    ----------
    file_paths : list of str
        Paths to Excel files.
    x_col : str
        Column name for x-axis values.
    y_cols : list of str
        Columns to analyze against x_col.
    """

    if y_cols is None:
        y_cols = [
            'Stroke Work',
            'Stroke Volume',
            'Ejection Fraction',
            'Heart Rate',
            'Cardiac Output'
        ]

    # Define exponential function
    def exp_func(x, a, b, c):
        return a * np.exp(b * x) + c

    tick_positions = [0.0, 1/3, 2/3, 1.0]
    tick_labels = ['0', '1/3', '2/3', '1']

    for file_path in file_paths:
        print(f"\n=== Processing {file_path} ===")
        df = pd.read_excel(file_path)

        # Normalize x-axis
        x_min, x_max = df[x_col].min(), df[x_col].max()
        df['Normalized X'] = (df[x_col] - x_min) / (x_max - x_min)
        grouped = df.groupby('Normalized X')

        for y_col in y_cols:
            if y_col not in df.columns:
                print(f"Column '{y_col}' not found in {file_path}. Skipping.")
                continue

            means = grouped[y_col].mean()
            stds = grouped[y_col].std()
            x_vals = means.index

            valid = ~(means.isna() | stds.isna())
            x_clean = x_vals[valid]
            y_mean = means[valid]
            y_std = stds[valid]

            if len(x_clean) < 3:
                print(f"Not enough data for {y_col} in {file_path}.")
                continue

            # ---- Linear & Quadratic fits ----
            linear_coeffs = np.polyfit(x_clean, y_mean, 1)
            quad_coeffs = np.polyfit(x_clean, y_mean, 2)

            y_linear_pred = np.polyval(linear_coeffs, x_clean)
            y_quad_pred = np.polyval(quad_coeffs, x_clean)

            ss_tot = np.sum((y_mean - np.mean(y_mean)) ** 2)
            r2_linear = 1 - np.sum((y_mean - y_linear_pred) ** 2) / ss_tot
            r2_quad = 1 - np.sum((y_mean - y_quad_pred) ** 2) / ss_tot

            # ---- Exponential fit ----
            try:
                popt, _ = curve_fit(exp_func, x_clean, y_mean, maxfev=5000)
                y_exp_pred = exp_func(x_clean, *popt)
                r2_exp = 1 - np.sum((y_mean - y_exp_pred) ** 2) / ss_tot
            except RuntimeError:
                popt = None
                r2_exp = np.nan

            # Smooth x for plotting fits
            x_fit = np.linspace(0, 1, 200)
            y_linear_fit = np.polyval(linear_coeffs, x_fit)
            y_quad_fit = np.polyval(quad_coeffs, x_fit)
            y_exp_fit = exp_func(x_fit, *popt) if popt is not None else None

            # ---- Plot ----
            plt.figure(figsize=(8, 6))
            plt.errorbar(x_clean, y_mean, yerr=y_std, fmt='o', 
                         label='Mean ± SD', color='black', capsize=5)
            plt.plot(x_fit, y_linear_fit, '--', label=f'Linear (R²={r2_linear:.3f})', color='blue')
            plt.plot(x_fit, y_quad_fit, '-.', label=f'Quadratic (R²={r2_quad:.3f})', color='red')
            if y_exp_fit is not None:
                plt.plot(x_fit, y_exp_fit, ':', label=f'Exponential (R²={r2_exp:.3f})', color='green')

            plt.xlabel(x_col)
            plt.ylabel(y_col)
            plt.title(f'{y_col} vs {x_col}\n{file_path}')
            plt.grid(True, which='both', linestyle='--', linewidth=0.5)
            plt.xticks(tick_positions, tick_labels)
            plt.legend()
            plt.tight_layout()
            plt.show()

            # ---- Print equations ----
            print(f"\n--- {y_col} ---")
            print(f"Linear fit:    y = {linear_coeffs[0]:.4f}x + {linear_coeffs[1]:.4f}     R² = {r2_linear:.4f}")
            print(f"Quadratic fit: y = {quad_coeffs[0]:.4f}x² + {quad_coeffs[1]:.4f}x + {quad_coeffs[2]:.4f}     R² = {r2_quad:.4f}")
            if popt is not None:
                print(f"Exponential fit: y = {popt[0]:.4f} * exp({popt[1]:.4f}x) + {popt[2]:.4f}     R² = {r2_exp:.4f}")



In [ ]:
files = [
    'PV_Fractional_Increase_T0-1_cleaned.xlsx',
    'PV_Performance_cleaned.xlsx',
]
analyze_files(files)


In [ ]:
file_path = 'PV_Fractional_Increase_T0-1_cleaned.xlsx'
df = pd.read_excel(file_path)

# Define x and y columns
x_col = 'Hemorrhage Rate %/Time'
y_cols = [
    'Stroke Work',
    'Stroke Volume',
    'Ejection Fraction',
    'Heart Rate',
    'Cardiac Output'
]

# Normalize x-axis values to 0–1 range
x_min = df[x_col].min()
x_max = df[x_col].max()
df['Normalized X'] = (df[x_col] - x_min) / (x_max - x_min)

# Group by normalized x and calculate mean/std
grouped = df.groupby('Normalized X')

# Define custom tick positions and labels
tick_positions = [0.0, 1/3, 2/3, 1.0]
# tick_labels = ['0', '1/3', '2/3', '3/3']
tick_labels = ['0', '1/3', '2/3', '1']

# Loop through each y-axis column
for y_col in y_cols:
    if y_col in df.columns:
        means = grouped[y_col].mean()
        stds = grouped[y_col].std()
        x_vals = means.index

        # Drop NaNs
        valid = ~(means.isna() | stds.isna())
        x_clean = x_vals[valid]
        y_mean = means[valid]
        y_std = stds[valid]

        # Fit models on normalized x
        linear_coeffs = np.polyfit(x_clean, y_mean, 1)
        quad_coeffs = np.polyfit(x_clean, y_mean, 2)

        y_linear_pred = np.polyval(linear_coeffs, x_clean)
        y_quad_pred = np.polyval(quad_coeffs, x_clean)

        ss_tot = np.sum((y_mean - np.mean(y_mean)) ** 2)
        r2_linear = 1 - np.sum((y_mean - y_linear_pred) ** 2) / ss_tot
        r2_quad = 1 - np.sum((y_mean - y_quad_pred) ** 2) / ss_tot

        # Smooth x for fit lines
        x_fit = np.linspace(0, 1, 200)
        y_linear_fit = np.polyval(linear_coeffs, x_fit)
        y_quad_fit = np.polyval(quad_coeffs, x_fit)

        # Plot
        plt.figure(figsize=(8, 6))
        plt.errorbar(x_clean, y_mean, yerr=y_std, fmt='o', label='Mean ± SD', color='black', capsize=5)
        plt.plot(x_fit, y_linear_fit, label=f'Linear Fit (R²={r2_linear:.3f})', linestyle='--', color='blue')
        plt.plot(x_fit, y_quad_fit, label=f'Quadratic Fit (R²={r2_quad:.3f})', linestyle='-.', color='red')

        plt.xlabel('Hemorrhage Rate %/Time')
        plt.ylabel(y_col)
        plt.title(f'{y_col} vs Hemorrhage Rate %/Time')
        plt.grid(True, which='both', linestyle='--', linewidth=0.5)
        plt.xticks(tick_positions, tick_labels)
        plt.legend()
        plt.tight_layout()
        plt.show()

        # Print equations and R²
        print(f"--- {y_col} ---")
        print(f"Linear fit:    y = {linear_coeffs[0]:.4f}x + {linear_coeffs[1]:.4f}     R² = {r2_linear:.4f}")
        print(f"Quadratic fit: y = {quad_coeffs[0]:.4f}x² + {quad_coeffs[1]:.4f}x + {quad_coeffs[2]:.4f}     R² = {r2_quad:.4f}")
        print()

    else:
        print(f"Column '{y_col}' not found in the data.")


In [ ]:
print('Done with the code being run.')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
import plotly.express as px

def analyze_and_plot(file_path, x_col, y_cols):
    """
    Analyze and plot mean/median trends with linear, quadratic, and exponential fits,
    alongside scatter + boxplot visualization. Also saves interactive scatter plots with Plotly.
    """

    # Load dataset
    df = pd.read_excel(file_path)

    # Normalize x-axis values
    x_min, x_max = df[x_col].min(), df[x_col].max()
    df['Normalized X'] = (df[x_col] - x_min) / (x_max - x_min)

    # Define custom ticks
    tick_positions = [0.0, 1/3, 2/3, 1.0]
    tick_labels = ['0', '1/3', '2/3', '1']

    # Define exponential function
    def exp_func(x, a, b, c):
        return a * np.exp(b * x) + c

    # Loop through each y-col
    for y_col in y_cols:
        if y_col not in df.columns:
            print(f"Column '{y_col}' not found in the data.")
            continue

        # Group by normalized x
        grouped = df.groupby('Normalized X')
        means = grouped[y_col].mean()
        medians = grouped[y_col].median()
        stds = grouped[y_col].std()

        x_vals = means.index

        # Drop NaNs
        valid = ~(means.isna() | stds.isna() | medians.isna())
        x_clean = x_vals[valid]
        y_mean = means[valid]
        y_median = medians[valid]
        y_std = stds[valid]

        # --- Function to fit and plot (mean or median separately) ---
        def fit_and_plot(y_data, label_color, label_name):
            # Linear
            linear_coeffs = np.polyfit(x_clean, y_data, 1)
            y_linear_pred = np.polyval(linear_coeffs, x_clean)

            # Quadratic
            quad_coeffs = np.polyfit(x_clean, y_data, 2)
            y_quad_pred = np.polyval(quad_coeffs, x_clean)

            # Exponential
            try:
                popt, _ = curve_fit(exp_func, x_clean, y_data, maxfev=10000)
                y_exp_pred = exp_func(x_clean, *popt)
                r2_exp = 1 - np.sum((y_data - y_exp_pred) ** 2) / np.sum((y_data - np.mean(y_data)) ** 2)
            except RuntimeError:
                popt = [np.nan, np.nan, np.nan]
                r2_exp = np.nan

            # R²
            ss_tot = np.sum((y_data - np.mean(y_data)) ** 2)
            r2_linear = 1 - np.sum((y_data - y_linear_pred) ** 2) / ss_tot
            r2_quad = 1 - np.sum((y_data - y_quad_pred) ** 2) / ss_tot

            # Plotting
            x_fit = np.linspace(0, 1, 200)
            plt.figure(figsize=(9, 7))

            # Boxplot with matplotlib for transparency
            data_by_bin = [df[df["Normalized X"] == xv][y_col].dropna().values for xv in sorted(df["Normalized X"].unique())]
            plt.boxplot(data_by_bin, positions=sorted(df["Normalized X"].unique()), widths=0.03,
                        patch_artist=True, boxprops=dict(facecolor="lightgray", alpha=0.4))

            # Scatter raw data on top
            plt.scatter(
                df['Normalized X'], df[y_col],
                alpha=0.6, s=25, color="black", label="Raw Data", zorder=3
            )

            # Fits
            plt.plot(x_fit, np.polyval(linear_coeffs, x_fit),
                     linestyle="--", color=label_color,
                     label=f"Linear (R²={r2_linear:.3f})", zorder=4)

            plt.plot(x_fit, np.polyval(quad_coeffs, x_fit),
                     linestyle="-.", color=label_color,
                     label=f"Quadratic (R²={r2_quad:.3f})", zorder=4)

            if not np.isnan(popt[0]):
                plt.plot(x_fit, exp_func(x_fit, *popt),
                         linestyle=":", color=label_color,
                         label=f"Exp (R²={r2_exp:.3f})", zorder=4)

            # Labels & Formatting
            plt.xlabel('Hemorrhage Rate %/Time (Normalized)')
            plt.ylabel(y_col)
            plt.title(f'{y_col} vs Hemorrhage Rate %/Time ({label_name})')
            plt.grid(True, linestyle="--", alpha=0.6)
            plt.xticks(tick_positions, tick_labels)
            plt.xlim(-0.05, 1.05)  # tighten x-axis spacing
            plt.legend()
            plt.tight_layout()
            plt.show()

            # Print equations
            print(f"--- {y_col} ({label_name}) ---")
            print(f"Linear:    y = {linear_coeffs[0]:.4f}x + {linear_coeffs[1]:.4f}     R² = {r2_linear:.4f}")
            print(f"Quadratic: y = {quad_coeffs[0]:.4f}x² + {quad_coeffs[1]:.4f}x + {quad_coeffs[2]:.4f}     R² = {r2_quad:.4f}")
            if not np.isnan(popt[0]):
                print(f"Exp:       y = {popt[0]:.4f} * exp({popt[1]:.4f}x) + {popt[2]:.4f}     R² = {r2_exp:.4f}")
            print()

        # --- Call separately for mean and median ---
        fit_and_plot(y_mean, "blue", "Mean")
        fit_and_plot(y_median, "red", "Median")

        # --- Plotly scatter export ---
        if "Parent Folder" in df.columns:
            fig = px.scatter(
                df, x="Normalized X", y=y_col,
                hover_data=["Parent Folder"],
                title=f"{y_col} Scatter Plot (Interactive)"
            )
            fig.update_traces(marker=dict(size=7, opacity=0.6))
            fig.write_html(f"{y_col}_scatter.html")
            fig.write_image(f"{y_col}_scatter.png")
            print(f"Saved Plotly scatter plots for {y_col} → HTML + PNG")


In [ ]:
x_col = 'Hemorrhage Rate %/Time'
y_cols = ['Stroke Work', 'Stroke Volume', 'Ejection Fraction', 'Heart Rate', 'Cardiac Output']

In [ ]:
analyze_and_plot('PV_Fractional_Increase_T0-1_cleaned.xlsx', x_col, y_cols)

In [ ]:
analyze_and_plot('PV_Performance_cleaned.xlsx', x_col, y_cols)

# Standard Deviation Edit

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
import plotly.express as px

def analyze_and_plot(file_path, x_col, y_cols):
    """
    Analyze and plot mean/median trends with linear, quadratic, and exponential fits,
    alongside scatter + boxplot visualization. Outliers beyond 2 std dev from the
    mean/median are removed from raw data before computing stats.
    """

    df = pd.read_excel(file_path)

    # Normalize x-axis
    x_min, x_max = df[x_col].min(), df[x_col].max()
    df['Normalized X'] = (df[x_col] - x_min) / (x_max - x_min)

    tick_positions = [0.0, 1/3, 2/3, 1.0]
    tick_labels = ['0', '1/3', '2/3', '1']

    def exp_func(x, a, b, c):
        return a * np.exp(b * x) + c

    for y_col in y_cols:
        if y_col not in df.columns:
            print(f"Column '{y_col}' not found in the data.")
            continue

        # --- Function to remove outliers and compute grouped stats ---
        def get_grouped_stats(df_sub, use_median=False):
            grouped_stats = []
            for x_val, group in df_sub.groupby('Normalized X'):
                y_vals = group[y_col].dropna()
                if len(y_vals) == 0:
                    continue
                if use_median:
                    center = np.median(y_vals)
                else:
                    center = np.mean(y_vals)
                std_dev = np.std(y_vals)
                # Keep only values within 2 std dev
                filtered = y_vals[(y_vals >= center - 2*std_dev) & (y_vals <= center + 2*std_dev)]
                grouped_stats.append((x_val, filtered.mean() if len(filtered) > 0 else np.nan,
                                      filtered.median() if len(filtered) > 0 else np.nan,
                                      filtered.std() if len(filtered) > 1 else np.nan))
            stats_df = pd.DataFrame(grouped_stats, columns=['Normalized X', 'Mean', 'Median', 'Std'])
            stats_df = stats_df.dropna()
            return stats_df

        stats_mean = get_grouped_stats(df, use_median=False)
        stats_median = get_grouped_stats(df, use_median=True)

        def fit_and_plot(stats_df, label_color, label_name, y_col_name):
            x_clean = stats_df['Normalized X']
            y_data = stats_df[y_col_name]

            if len(y_data) < 3:
                print(f"Not enough points for {y_col} ({label_name}) after outlier removal")
                return

            # Linear
            linear_coeffs = np.polyfit(x_clean, y_data, 1)
            y_linear_pred = np.polyval(linear_coeffs, x_clean)

            # Quadratic
            quad_coeffs = np.polyfit(x_clean, y_data, 2)
            y_quad_pred = np.polyval(quad_coeffs, x_clean)

            # Exponential
            try:
                popt, _ = curve_fit(exp_func, x_clean, y_data, maxfev=10000)
                y_exp_pred = exp_func(x_clean, *popt)
                r2_exp = 1 - np.sum((y_data - y_exp_pred)**2) / np.sum((y_data - np.mean(y_data))**2)
            except RuntimeError:
                popt = [np.nan, np.nan, np.nan]
                r2_exp = np.nan

            ss_tot = np.sum((y_data - np.mean(y_data))**2)
            r2_linear = 1 - np.sum((y_data - y_linear_pred)**2) / ss_tot
            r2_quad = 1 - np.sum((y_data - y_quad_pred)**2) / ss_tot

            # Plot
            x_fit = np.linspace(0, 1, 200)
            plt.figure(figsize=(9,7))

            # Boxplot: still show raw data
            data_by_bin = [df[df["Normalized X"] == xv][y_col].dropna().values for xv in sorted(df["Normalized X"].unique())]
            plt.boxplot(data_by_bin, positions=sorted(df["Normalized X"].unique()), widths=0.03,
                        patch_artist=True, boxprops=dict(facecolor="lightgray", alpha=0.4))

            plt.scatter(df['Normalized X'], df[y_col], alpha=0.6, s=25, color="black", label="Raw Data", zorder=3)
            plt.plot(x_fit, np.polyval(linear_coeffs, x_fit), linestyle="--", color=label_color, label=f"Linear (R²={r2_linear:.3f})", zorder=4)
            plt.plot(x_fit, np.polyval(quad_coeffs, x_fit), linestyle="-.", color=label_color, label=f"Quadratic (R²={r2_quad:.3f})", zorder=4)
            if not np.isnan(popt[0]):
                plt.plot(x_fit, exp_func(x_fit, *popt), linestyle=":", color=label_color, label=f"Exp (R²={r2_exp:.3f})", zorder=4)

            plt.xlabel('Hemorrhage Rate %/Time (Normalized)')
            plt.ylabel(y_col)
            plt.title(f'{y_col} vs Hemorrhage Rate %/Time ({label_name}, Outliers Removed)')
            plt.grid(True, linestyle="--", alpha=0.6)
            plt.xticks(tick_positions, tick_labels)
            plt.xlim(-0.05, 1.05)
            plt.legend()
            plt.tight_layout()
            plt.show()

            # Print equations
            print(f"--- {y_col} ({label_name}) ---")
            print(f"Linear:    y = {linear_coeffs[0]:.4f}x + {linear_coeffs[1]:.4f}     R² = {r2_linear:.4f}")
            print(f"Quadratic: y = {quad_coeffs[0]:.4f}x² + {quad_coeffs[1]:.4f}x + {quad_coeffs[2]:.4f}     R² = {r2_quad:.4f}")
            if not np.isnan(popt[0]):
                print(f"Exp:       y = {popt[0]:.4f} * exp({popt[1]:.4f}x) + {popt[2]:.4f}     R² = {r2_exp:.4f}")
            print()

        # Plot mean-based fits
        fit_and_plot(stats_mean, "blue", "Mean", "Mean")
        # Plot median-based fits
        fit_and_plot(stats_median, "red", "Median", "Median")

        # --- Plotly scatter export ---
        if "Parent Folder" in df.columns:
            fig = px.scatter(df, x="Normalized X", y=y_col, hover_data=["Parent Folder"], title=f"{y_col} Scatter Plot (Interactive)")
            fig.update_traces(marker=dict(size=7, opacity=0.6))
            fig.write_html(f"{y_col}_scatter.html")
            fig.write_image(f"{y_col}_scatter.png")
            print(f"Saved Plotly scatter plots for {y_col} → HTML + PNG")


In [ ]:
analyze_and_plot('PV_Fractional_Increase_T0-1_cleaned.xlsx', x_col, y_cols)

In [ ]:
analyze_and_plot('PV_Performance_cleaned.xlsx', x_col, y_cols)

# HEMORRHAGE LEVEL STUFF


In [ ]:
# # import pandas as pd
# # import matplotlib.pyplot as plt
# # import numpy as np
# # from scipy.optimize import curve_fit
# # import plotly.express as px

# # def analyze_and_plot(file_path, x_col, y_cols):
# #     """
# #     Analyze and plot mean/median trends with linear, quadratic, and exponential fits,
# #     alongside scatter + boxplot visualization. Outliers beyond 2 std dev from the
# #     mean/median are removed from raw data before computing stats.
# #     """

# #     df = pd.read_excel(file_path)

# #     # Normalize x-axis
# #     x_min, x_max = df[x_col].min(), df[x_col].max()
# #     df['Normalized X'] = (df[x_col] - x_min) / (x_max - x_min)

# #     tick_positions = [0.0, 10, 20, 30]
# #     tick_labels = ['0', '1/3', '2/3', '1']

# #     def exp_func(x, a, b, c):
# #         return a * np.exp(b * x) + c

# #     for y_col in y_cols:
# #         if y_col not in df.columns:
# #             print(f"Column '{y_col}' not found in the data.")
# #             continue

# #         # --- Function to remove outliers and compute grouped stats ---
# #         def get_grouped_stats(df_sub, use_median=False):
# #             grouped_stats = []
# #             for x_val, group in df_sub.groupby('Normalized X'):
# #                 y_vals = group[y_col].dropna()
# #                 if len(y_vals) == 0:
# #                     continue
# #                 if use_median:
# #                     center = np.median(y_vals)
# #                 else:
# #                     center = np.mean(y_vals)
# #                 std_dev = np.std(y_vals)
# #                 # Keep only values within 2 std dev
# #                 filtered = y_vals[(y_vals >= center - 2*std_dev) & (y_vals <= center + 2*std_dev)]
# #                 grouped_stats.append((x_val, filtered.mean() if len(filtered) > 0 else np.nan,
# #                                       filtered.median() if len(filtered) > 0 else np.nan,
# #                                       filtered.std() if len(filtered) > 1 else np.nan))
# #             stats_df = pd.DataFrame(grouped_stats, columns=['Normalized X', 'Mean', 'Median', 'Std'])
# #             stats_df = stats_df.dropna()
# #             return stats_df

# #         stats_mean = get_grouped_stats(df, use_median=False)
# #         stats_median = get_grouped_stats(df, use_median=True)

# #         def fit_and_plot(stats_df, label_color, label_name, y_col_name):
# #             x_clean = stats_df['Normalized X']
# #             y_data = stats_df[y_col_name]

# #             if len(y_data) < 3:
# #                 print(f"Not enough points for {y_col} ({label_name}) after outlier removal")
# #                 return

# #             # Linear
# #             linear_coeffs = np.polyfit(x_clean, y_data, 1)
# #             y_linear_pred = np.polyval(linear_coeffs, x_clean)

# #             # Quadratic
# #             quad_coeffs = np.polyfit(x_clean, y_data, 2)
# #             y_quad_pred = np.polyval(quad_coeffs, x_clean)

# #             # Exponential
# #             try:
# #                 popt, _ = curve_fit(exp_func, x_clean, y_data, maxfev=10000)
# #                 y_exp_pred = exp_func(x_clean, *popt)
# #                 r2_exp = 1 - np.sum((y_data - y_exp_pred)**2) / np.sum((y_data - np.mean(y_data))**2)
# #             except RuntimeError:
# #                 popt = [np.nan, np.nan, np.nan]
# #                 r2_exp = np.nan

# #             ss_tot = np.sum((y_data - np.mean(y_data))**2)
# #             r2_linear = 1 - np.sum((y_data - y_linear_pred)**2) / ss_tot
# #             r2_quad = 1 - np.sum((y_data - y_quad_pred)**2) / ss_tot

# #             # Plot
# #             x_fit = np.linspace(0, 1, 200)
# #             plt.figure(figsize=(9,7))

# #             # Boxplot: still show raw data
# #             data_by_bin = [df[df["Normalized X"] == xv][y_col].dropna().values for xv in sorted(df["Normalized X"].unique())]
# #             plt.boxplot(data_by_bin, positions=sorted(df["Normalized X"].unique()), widths=0.03,
# #                         patch_artist=True, boxprops=dict(facecolor="lightgray", alpha=0.4))

# #             plt.scatter(df['Normalized X'], df[y_col], alpha=0.6, s=25, color="black", label="Raw Data", zorder=3)
# #             plt.plot(x_fit, np.polyval(linear_coeffs, x_fit), linestyle="--", color=label_color, label=f"Linear (R²={r2_linear:.3f})", zorder=4)
# #             plt.plot(x_fit, np.polyval(quad_coeffs, x_fit), linestyle="-.", color=label_color, label=f"Quadratic (R²={r2_quad:.3f})", zorder=4)
# #             if not np.isnan(popt[0]):
# #                 plt.plot(x_fit, exp_func(x_fit, *popt), linestyle=":", color=label_color, label=f"Exp (R²={r2_exp:.3f})", zorder=4)

# #             plt.xlabel('Hemorrhage Level (Normalized)')
# #             plt.ylabel(y_col)
# #             plt.title(f'{y_col} vs Hemorrhage Level ({label_name}, Outliers Removed)')
# #             plt.grid(True, linestyle="--", alpha=0.6)
# #             plt.xticks(tick_positions, tick_labels)
# #             plt.xlim(-0.05, 1.05)
# #             plt.legend()
# #             plt.tight_layout()
# #             plt.show()

# #             # Print equations
# #             print(f"--- {y_col} ({label_name}) ---")
# #             print(f"Linear:    y = {linear_coeffs[0]:.4f}x + {linear_coeffs[1]:.4f}     R² = {r2_linear:.4f}")
# #             print(f"Quadratic: y = {quad_coeffs[0]:.4f}x² + {quad_coeffs[1]:.4f}x + {quad_coeffs[2]:.4f}     R² = {r2_quad:.4f}")
# #             if not np.isnan(popt[0]):
# #                 print(f"Exp:       y = {popt[0]:.4f} * exp({popt[1]:.4f}x) + {popt[2]:.4f}     R² = {r2_exp:.4f}")
# #             print()

# #         # Plot mean-based fits
# #         fit_and_plot(stats_mean, "blue", "Mean", "Mean")
# #         # Plot median-based fits
# #         fit_and_plot(stats_median, "red", "Median", "Median")

# #         # --- Plotly scatter export ---
# #         if "Parent Folder" in df.columns:
# #             fig = px.scatter(df, x="Normalized X", y=y_col, hover_data=["Parent Folder"], title=f"{y_col} Scatter Plot (Interactive)")
# #             fig.update_traces(marker=dict(size=7, opacity=0.6))
# #             fig.write_html(f"{y_col}_scatter.html")
# #             fig.write_image(f"{y_col}_scatter.png")
# #             print(f"Saved Plotly scatter plots for {y_col} → HTML + PNG")

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
import plotly.express as px

def analyze_and_plot(file_path, y_cols):
    """
    Analyze and plot mean/median trends with linear, quadratic, and exponential fits,
    alongside scatter + boxplot visualization.
    Outliers beyond 2 std dev from the mean/median are removed from raw data before computing stats.
    X-axis = Hemorrhage Level (0, 10, 20, 30).
    Points are colored by 'Hemorrhage Rate %/Time'.
    """

    df = pd.read_excel(file_path)

    x_col = "Hemorrhage Level"   # Fixed axis column
    rate_col = "Hemorrhage Rate %/Time"  # Column for coloring

    # Define x ticks
    tick_positions = [0, 10, 20, 30]
    tick_labels = [str(val) for val in tick_positions]

    def exp_func(x, a, b, c):
        return a * np.exp(b * x) + c

    for y_col in y_cols:
        if y_col not in df.columns:
            print(f"Column '{y_col}' not found in the data.")
            continue

        # --- Function to remove outliers and compute grouped stats ---
        def get_grouped_stats(df_sub, use_median=False):
            grouped_stats = []
            for x_val, group in df_sub.groupby(x_col):
                y_vals = group[y_col].dropna()
                if len(y_vals) == 0:
                    continue
                if use_median:
                    center = np.median(y_vals)
                else:
                    center = np.mean(y_vals)
                std_dev = np.std(y_vals)
                # Keep only values within 2 std dev
                filtered = y_vals[(y_vals >= center - 2*std_dev) & (y_vals <= center + 2*std_dev)]
                grouped_stats.append((x_val, 
                                      filtered.mean() if len(filtered) > 0 else np.nan,
                                      filtered.median() if len(filtered) > 0 else np.nan,
                                      filtered.std() if len(filtered) > 1 else np.nan))
            stats_df = pd.DataFrame(grouped_stats, columns=[x_col, 'Mean', 'Median', 'Std'])
            stats_df = stats_df.dropna()
            return stats_df

        stats_mean = get_grouped_stats(df, use_median=False)
        stats_median = get_grouped_stats(df, use_median=True)

        def fit_and_plot(stats_df, label_color, label_name, y_col_name):
            x_clean = stats_df[x_col]
            y_data = stats_df[y_col_name]

            if len(y_data) < 3:
                print(f"Not enough points for {y_col} ({label_name}) after outlier removal")
                return

            # Linear
            linear_coeffs = np.polyfit(x_clean, y_data, 1)
            y_linear_pred = np.polyval(linear_coeffs, x_clean)

            # Quadratic
            quad_coeffs = np.polyfit(x_clean, y_data, 2)
            y_quad_pred = np.polyval(quad_coeffs, x_clean)

            # Exponential
            try:
                popt, _ = curve_fit(exp_func, x_clean, y_data, maxfev=10000)
                y_exp_pred = exp_func(x_clean, *popt)
                r2_exp = 1 - np.sum((y_data - y_exp_pred)**2) / np.sum((y_data - np.mean(y_data))**2)
            except RuntimeError:
                popt = [np.nan, np.nan, np.nan]
                r2_exp = np.nan

            ss_tot = np.sum((y_data - np.mean(y_data))**2)
            r2_linear = 1 - np.sum((y_data - y_linear_pred)**2) / ss_tot
            r2_quad = 1 - np.sum((y_data - y_quad_pred)**2) / ss_tot

            # Plot
            x_fit = np.linspace(min(x_clean), max(x_clean), 200)
            plt.figure(figsize=(9,7))

            # Boxplot: show raw data grouped by Hemorrhage Level
            data_by_bin = [df[df[x_col] == xv][y_col].dropna().values 
                           for xv in sorted(df[x_col].unique())]
            plt.boxplot(data_by_bin, positions=sorted(df[x_col].unique()), widths=1.5,
                        patch_artist=True, boxprops=dict(facecolor="lightgray", alpha=0.4))

            # Scatter colored by Hemorrhage Rate %/Time
            if rate_col in df.columns:
                sc = plt.scatter(df[x_col], df[y_col],
                                 c=df[rate_col],
                                 cmap="viridis",
                                 alpha=0.7, s=40, edgecolor="k", zorder=3)
                cbar = plt.colorbar(sc)
                cbar.set_label(rate_col)
            else:
                plt.scatter(df[x_col], df[y_col],
                            alpha=0.6, s=25, color="black", label="Raw Data", zorder=3)

            # Fit lines
            plt.plot(x_fit, np.polyval(linear_coeffs, x_fit), linestyle="--", color=label_color, 
                     label=f"Linear (R²={r2_linear:.3f})", zorder=4)
            plt.plot(x_fit, np.polyval(quad_coeffs, x_fit), linestyle="-.", color=label_color, 
                     label=f"Quadratic (R²={r2_quad:.3f})", zorder=4)
            if not np.isnan(popt[0]):
                plt.plot(x_fit, exp_func(x_fit, *popt), linestyle=":", color=label_color, 
                         label=f"Exp (R²={r2_exp:.3f})", zorder=4)

            plt.xlabel('Hemorrhage Level')
            plt.ylabel(y_col)
            plt.title(f'{y_col} vs Hemorrhage Level ({label_name}, Outliers Removed)')
            plt.grid(True, linestyle="--", alpha=0.6)
            plt.xticks(tick_positions, tick_labels)
            plt.xlim(-2, 32)
            plt.legend(loc="upper left")  # Legend for fits
            plt.tight_layout()
            plt.show()

            # Print equations
            print(f"--- {y_col} ({label_name}) ---")
            print(f"Linear:    y = {linear_coeffs[0]:.4f}x + {linear_coeffs[1]:.4f}     R² = {r2_linear:.4f}")
            print(f"Quadratic: y = {quad_coeffs[0]:.4f}x² + {quad_coeffs[1]:.4f}x + {quad_coeffs[2]:.4f}     R² = {r2_quad:.4f}")
            if not np.isnan(popt[0]):
                print(f"Exp:       y = {popt[0]:.4f} * exp({popt[1]:.4f}x) + {popt[2]:.4f}     R² = {r2_exp:.4f}")
            print()

        # Plot mean-based fits
        fit_and_plot(stats_mean, "blue", "Mean", "Mean")
        # Plot median-based fits
        fit_and_plot(stats_median, "red", "Median", "Median")

        # --- Plotly scatter export ---
        if "Parent Folder" in df.columns:
            color_arg = rate_col if rate_col in df.columns else None
            fig = px.scatter(df, x=x_col, y=y_col, color=color_arg,
                             hover_data=["Parent Folder"], 
                             title=f"{y_col} Scatter Plot (Interactive)",
                             color_continuous_scale="viridis")
            fig.update_traces(marker=dict(size=7, opacity=0.6))
            fig.write_html(f"{y_col}_scatter.html")
            fig.write_image(f"{y_col}_scatter.png")
            print(f"Saved Plotly scatter plots for {y_col} → HTML + PNG")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
import plotly.express as px
import matplotlib.patches as mpatches

def analyze_and_plot(file_path, y_cols):
    """
    Analyze and plot mean/median trends with linear, quadratic, and exponential fits,
    alongside scatter + boxplot visualization.
    Outliers beyond 2 std dev from the mean/median are removed from raw data before computing stats.
    X-axis = Hemorrhage Level (0, 10, 20, 30).
    Points are colored by 'Hemorrhage Rate %/Time'.
    """

    df = pd.read_excel(file_path)

    x_col = "True Hemorrhage Level"
    rate_col = "Hemorrhage Rate %/Time"

    tick_positions = [0, 10, 20, 30]
    tick_labels = [str(val) for val in tick_positions]

    def exp_func(x, a, b, c):
        return a * np.exp(b * x) + c

    for y_col in y_cols:
        if y_col not in df.columns:
            print(f"Column '{y_col}' not found in the data.")
            continue

        # --- Function to remove outliers and compute grouped stats ---
        def get_grouped_stats(df_sub, use_median=False):
            grouped_stats = []
            for x_val, group in df_sub.groupby(x_col):
                y_vals = group[y_col].dropna()
                if len(y_vals) == 0:
                    continue
                center = np.median(y_vals) if use_median else np.mean(y_vals)
                std_dev = np.std(y_vals)
                filtered = y_vals[(y_vals >= center - 2*std_dev) & (y_vals <= center + 2*std_dev)]
                grouped_stats.append((x_val,
                                      filtered.mean() if len(filtered) > 0 else np.nan,
                                      filtered.median() if len(filtered) > 0 else np.nan,
                                      filtered.std() if len(filtered) > 1 else np.nan))
            stats_df = pd.DataFrame(grouped_stats, columns=[x_col, 'Mean', 'Median', 'Std'])
            return stats_df.dropna()

        stats_mean = get_grouped_stats(df, use_median=False)
        stats_median = get_grouped_stats(df, use_median=True)

        def fit_and_plot(stats_df, label_color, label_name, y_col_name):
            x_clean = stats_df[x_col]
            y_data = stats_df[y_col_name]

            if len(y_data) < 3:
                print(f"Not enough points for {y_col} ({label_name}) after outlier removal")
                return

            # Fit lines
            linear_coeffs = np.polyfit(x_clean, y_data, 1)
            quad_coeffs = np.polyfit(x_clean, y_data, 2)
            try:
                popt, _ = curve_fit(exp_func, x_clean, y_data, maxfev=10000)
                y_exp_pred = exp_func(x_clean, *popt)
                r2_exp = 1 - np.sum((y_data - y_exp_pred)**2) / np.sum((y_data - np.mean(y_data))**2)
            except RuntimeError:
                popt = [np.nan, np.nan, np.nan]
                r2_exp = np.nan

            ss_tot = np.sum((y_data - np.mean(y_data))**2)
            r2_linear = 1 - np.sum((y_data - np.polyval(linear_coeffs, x_clean))**2)/ss_tot
            r2_quad = 1 - np.sum((y_data - np.polyval(quad_coeffs, x_clean))**2)/ss_tot

            x_fit = np.linspace(min(x_clean), max(x_clean), 200)
            plt.figure(figsize=(9,7))

            # Boxplot
            data_by_bin = [df[df[x_col]==xv][y_col].dropna().values for xv in sorted(df[x_col].unique())]
            plt.boxplot(data_by_bin, positions=sorted(df[x_col].unique()), widths=1.5,
                        patch_artist=True, boxprops=dict(facecolor="lightgray", alpha=0.4))

            # Scatter points colored by Hemorrhage Rate
            scatter_handles = []
            if rate_col in df.columns:
                sc = plt.scatter(df[x_col], df[y_col],
                                 c=df[rate_col],
                                 cmap="viridis",
                                 alpha=0.7, s=50, edgecolor="k", zorder=3)
                # Create discrete legend for the four possible rates
                rate_values = sorted(df[rate_col].unique())
                for val in rate_values:
                    color_val = plt.cm.viridis((val - min(rate_values))/(max(rate_values) - min(rate_values)))
                    scatter_handles.append(mpatches.Patch(color=color_val, label=f"Hemorrhage Rate {val}"))
            else:
                plt.scatter(df[x_col], df[y_col], color="black", alpha=0.7, s=50, zorder=3)

            # Fit lines
            plt.plot(x_fit, np.polyval(linear_coeffs, x_fit), linestyle="--", color=label_color,
                     label=f"Linear (R²={r2_linear:.3f})", zorder=4)
            plt.plot(x_fit, np.polyval(quad_coeffs, x_fit), linestyle="-.", color=label_color,
                     label=f"Quadratic (R²={r2_quad:.3f})", zorder=4)
            if not np.isnan(popt[0]):
                plt.plot(x_fit, exp_func(x_fit, *popt), linestyle=":", color=label_color,
                         label=f"Exp (R²={r2_exp:.3f})", zorder=4)

            plt.xlabel('Hemorrhage Level')
            plt.ylabel(y_col)
            plt.title(f'{y_col} vs Hemorrhage Level ({label_name}, Outliers Removed)')
            plt.grid(True, linestyle="--", alpha=0.6)
            plt.xticks(tick_positions, tick_labels)
            plt.xlim(-2, 32)

            # Combine scatter legend and fit lines
            line_handles, line_labels = plt.gca().get_legend_handles_labels()
            plt.legend(handles=scatter_handles + line_handles, loc="upper left")

            plt.tight_layout()
            plt.show()

            # Print equations
            print(f"--- {y_col} ({label_name}) ---")
            print(f"Linear:    y = {linear_coeffs[0]:.4f}x + {linear_coeffs[1]:.4f}     R² = {r2_linear:.4f}")
            print(f"Quadratic: y = {quad_coeffs[0]:.4f}x² + {quad_coeffs[1]:.4f}x + {quad_coeffs[2]:.4f}     R² = {r2_quad:.4f}")
            if not np.isnan(popt[0]):
                print(f"Exp:       y = {popt[0]:.4f} * exp({popt[1]:.4f}x) + {popt[2]:.4f}     R² = {r2_exp:.4f}")
            print()

        # Plot mean and median fits
        fit_and_plot(stats_mean, "blue", "Mean", "Mean")
        fit_and_plot(stats_median, "red", "Median", "Median")

        # Plotly scatter export
        if "Parent Folder" in df.columns:
            if rate_col in df.columns:
                fig = px.scatter(df, x=x_col, y=y_col, color=rate_col,
                                 color_continuous_scale="viridis",
                                 hover_data=["Parent Folder"],
                                 title=f"{y_col} Scatter Plot (Interactive)")
            else:
                fig = px.scatter(df, x=x_col, y=y_col, hover_data=["Parent Folder"],
                                 title=f"{y_col} Scatter Plot (Interactive)")
            fig.update_traces(marker=dict(size=7, opacity=0.6))
            fig.write_html(f"{y_col}_scatter.html")
            fig.write_image(f"{y_col}_scatter.png")
            print(f"Saved Plotly scatter plots for {y_col} → HTML + PNG")


In [ ]:
analyze_and_plot('PV_Fractional_Increase_T0-1_cleaned.xlsx', y_cols)

In [ ]:
analyze_and_plot('PV_Performance_cleaned.xlsx', y_cols)